In [0]:

USE CATALOG dbw_healthcare_de_dev;
USE SCHEMA silver;

CREATE OR REPLACE PROCEDURE usp_LoadDimPatients(
    IN p_FileId STRING
)
LANGUAGE SQL
SQL SECURITY INVOKER
AS
BEGIN
    -- ==========================================
    -- 1. Declare variables
    -- ==========================================

    DECLARE v_BatchId STRING;
    DECLARE v_FileCount, v_BronzeCount BIGINT;
    DECLARE v_DuplicateCount, v_InvalidCount BIGINT;

    -- ==========================================
    -- 2. Validate file eligibility
    -- ==========================================

    SET v_FileCount = (
        SELECT COUNT(*)
        FROM dbw_healthcare_de_dev.etl.file_control
        WHERE file_id = p_FileId
          AND processing_stage = 'SILVER'
          AND pipeline_status = 'IN_PROGRESS'
          AND is_enabled = true
    );

    IF v_FileCount <> 1 THEN
        SIGNAL SQLSTATE '45000'
        SET MESSAGE_TEXT =
            'File is not eligible for Silver processing';
    END IF;

    SET v_BatchId = (
        SELECT MAX(batch_id)
        FROM dbw_healthcare_de_dev.etl.file_control
        WHERE file_id = p_FileId
    );

    -- ==========================================
    -- 3. Validate Bronze records
    -- ==========================================

    SET v_BronzeCount = (
        SELECT COUNT(*)
        FROM dbw_healthcare_de_dev.bronze.patients_raw
        WHERE file_id = p_FileId
          AND batch_id = v_BatchId
    );

    IF v_BronzeCount = 0 THEN
        SIGNAL SQLSTATE '45000'
        SET MESSAGE_TEXT =
            'No matching Bronze records found';
    END IF;

    -- ==========================================
    -- 4. Reject duplicate source patients
    -- ==========================================

    SET v_DuplicateCount = (
        SELECT COUNT(*)
        FROM (
            SELECT patient_id
            FROM dbw_healthcare_de_dev.bronze.patients_raw
            WHERE file_id = p_FileId
              AND batch_id = v_BatchId
            GROUP BY patient_id
            HAVING COUNT(*) > 1
        ) AS Duplicates
    );

    IF v_DuplicateCount > 0 THEN
        SIGNAL SQLSTATE '45000'
        SET MESSAGE_TEXT =
            'Duplicate patients detected in source batch';
    END IF;

    -- ==========================================
    -- 5. Validate incoming chronological order
    -- ==========================================

    SET v_InvalidCount = (
        SELECT COUNT(*)
        FROM dbw_healthcare_de_dev.bronze.patients_raw B
        LEFT JOIN (
            SELECT
                PatientId,
                MAX(RowStartDate) AS LatestStart
            FROM dbw_healthcare_de_dev.silver.DimPatients
            GROUP BY PatientId
        ) D
            ON B.patient_id = D.PatientId
        WHERE B.file_id = p_FileId
          AND B.batch_id = v_BatchId
          AND (
              B.patient_id IS NULL
              OR B.ingestion_timestamp IS NULL
              OR (
                  D.LatestStart IS NOT NULL
                  AND B.ingestion_timestamp < D.LatestStart
              )
          )
    );

    IF v_InvalidCount > 0 THEN
        SIGNAL SQLSTATE '45000'
        SET MESSAGE_TEXT =
            'Invalid patient or out-of-order batch';
    END IF;

    -- ==========================================
    -- 6. Reject conflicting existing versions
    -- ==========================================

    SET v_InvalidCount = (
        SELECT COUNT(*)
        FROM dbw_healthcare_de_dev.bronze.patients_raw B
        JOIN dbw_healthcare_de_dev.silver.DimPatients D
          ON B.patient_id = D.PatientId
         AND B.ingestion_timestamp = D.RowStartDate
        WHERE B.file_id = p_FileId
          AND B.batch_id = v_BatchId
          AND (
              NOT (B.gender <=> D.Gender)
              OR NOT (B.date_of_birth <=> D.DateOfBirth)
              OR NOT (B.state <=> D.State)
              OR NOT (B.batch_id <=> D.BatchId)
          )
    );

    IF v_InvalidCount > 0 THEN
        SIGNAL SQLSTATE '45000'
        SET MESSAGE_TEXT =
            'Conflicting Silver version at incoming timestamp';
    END IF;

    -- ==========================================
    -- 7. SCD Type 2: expire and insert versions
    -- ==========================================

    MERGE INTO dbw_healthcare_de_dev.silver.DimPatients AS T
    USING (
        WITH Incoming AS (
            SELECT
                B.patient_id AS PatientId,
                B.first_name AS FirstName,
                B.last_name AS LastName,
                B.gender AS Gender,
                B.date_of_birth AS DateOfBirth,
                B.state AS State,
                B.batch_id AS BatchId,
                B.source_file AS SourceFile,
                B.ingestion_timestamp AS IngestionTimestamp,
                SHA2(
                    TO_JSON(
                        NAMED_STRUCT(
                            'Gender', B.gender,
                            'DateOfBirth', B.date_of_birth,
                            'State', B.state
                        )
                    ),
                    256
                ) AS RecordHash
            FROM dbw_healthcare_de_dev.bronze.patients_raw B
            WHERE B.file_id = p_FileId
              AND B.batch_id = v_BatchId
        ),
        Classified AS (
            SELECT
                I.*,
                D.PatientId AS ExistingPatientId,
                D.RecordHash AS ExistingHash,
                D.RowStartDate AS ExistingStartDate
            FROM Incoming I
            LEFT JOIN dbw_healthcare_de_dev.silver.DimPatients D
                ON I.PatientId = D.PatientId
               AND D.IsActive = true
        ),
        Staged AS (
            -- Expire the current active version
            SELECT
                PatientId AS MergeKey,
                'EXPIRE' AS MergeAction,
                *
            FROM Classified
            WHERE ExistingPatientId IS NOT NULL
              AND NOT (RecordHash <=> ExistingHash)
              AND IngestionTimestamp > ExistingStartDate

            UNION ALL

            -- Insert new patients or Type 2 versions
            SELECT
                CAST(NULL AS STRING) AS MergeKey,
                'INSERT' AS MergeAction,
                *
            FROM Classified
            WHERE ExistingPatientId IS NULL
               OR (
                   NOT (RecordHash <=> ExistingHash)
                   AND IngestionTimestamp > ExistingStartDate
               )
        )
        SELECT * FROM Staged
    ) AS S

    ON T.PatientId = S.MergeKey
       AND T.IsActive = true

    WHEN MATCHED
        AND S.MergeAction = 'EXPIRE'
    THEN UPDATE SET
        T.RowEndDate = S.IngestionTimestamp,
        T.IsActive = false

    WHEN NOT MATCHED
        AND S.MergeAction = 'INSERT'
    THEN INSERT (
        PatientId,
        FirstName,
        LastName,
        Gender,
        DateOfBirth,
        State,
        BatchId,
        SourceFile,
        IngestionTimestamp,
        RecordHash,
        RowStartDate,
        RowEndDate,
        IsActive
    )
    VALUES (
        S.PatientId,
        S.FirstName,
        S.LastName,
        S.Gender,
        S.DateOfBirth,
        S.State,
        S.BatchId,
        S.SourceFile,
        S.IngestionTimestamp,
        S.RecordHash,
        S.IngestionTimestamp,
        NULL,
        true
    );

    -- ==========================================
    -- 8. SCD Type 1: overwrite names
    --    across all historical versions
    -- ==========================================

    MERGE INTO dbw_healthcare_de_dev.silver.DimPatients AS T
    USING (
        SELECT
            B.patient_id AS PatientId,
            B.first_name AS FirstName,
            B.last_name AS LastName,
            B.ingestion_timestamp AS IncomingTimestamp
        FROM dbw_healthcare_de_dev.bronze.patients_raw B
        WHERE B.file_id = p_FileId
          AND B.batch_id = v_BatchId
          AND NOT EXISTS (
              SELECT 1
              FROM dbw_healthcare_de_dev.silver.DimPatients D
              WHERE D.PatientId = B.patient_id
                AND D.RowStartDate > B.ingestion_timestamp
          )
    ) AS S

    ON T.PatientId = S.PatientId

    WHEN MATCHED AND (
        NOT (T.FirstName <=> S.FirstName)
        OR NOT (T.LastName <=> S.LastName)
    )
    THEN UPDATE SET
        T.FirstName = S.FirstName,
        T.LastName = S.LastName;

    -- ==========================================
    -- 9. Validate active patient versions
    -- ==========================================

    SET v_InvalidCount = (
        SELECT COUNT(*)
        FROM (
            SELECT PatientId
            FROM dbw_healthcare_de_dev.silver.DimPatients
            GROUP BY PatientId
            HAVING SUM(
                CASE WHEN IsActive THEN 1 ELSE 0 END
            ) <> 1
        ) AS InvalidPatients
    );

    IF v_InvalidCount > 0 THEN
        SIGNAL SQLSTATE '45000'
        SET MESSAGE_TEXT =
            'Silver validation failed: invalid active versions';
    END IF;

    -- ==========================================
    -- 10. Return processing result
    -- ==========================================

    SELECT
        p_FileId AS FileId,
        v_BatchId AS BatchId,
        v_BronzeCount AS BronzeRecords,
        'SILVER_COMPLETED' AS Status;

END;
